In [1]:
import networkx as nx

def to_number(x):
    try:
        return int(x)
    except ValueError:
        return float(x)

def read_pajek(name, path = "."):
    names = dict()
    G = nx.MultiDiGraph()
    with open(path + "/" + name + ".net", 'r', encoding="utf-8") as file:
        file.readline()

        for line in file:
            if line.startswith("*"):
                break
            else:
                node = line.split("\"")
                G.add_node(int(node[0]) - 1, label = node[1])
                names[int(node[0]) - 1] = node[1]

        for line in file:
            i, j, w = map(to_number, line.split())
            i -= 1
            j -= 1
            G.add_edge(i, j, weight=float(w))
      
    return G, names



graphs = dict()
name_mapping = dict()
digraphs = dict()
modes = ["no_weights", "time_diff", "normalized_time_diff", "scaled_time_diff", "points", "pure_points"]
for mode in modes:
    graphs[mode], name_mapping[mode] = read_pajek(f"TDF_{mode}", "output_graphs")
    print(f"Graph Info: {mode}")


Graph Info: no_weights
Graph Info: time_diff
Graph Info: normalized_time_diff
Graph Info: scaled_time_diff
Graph Info: points
Graph Info: pure_points


In [2]:
results = dict()
for mode in modes:
    result = nx.pagerank(graphs[mode],max_iter=10000)
    results[mode] = {name: score for name,score in zip(name_mapping[mode].values(),result.values())}
    results[mode] = {k:v for k, v in sorted(results[mode].items(), key=lambda x: x[1], reverse=True)}


In [33]:
combined_scores = {k: 0 for k in results["points"].keys()}

for name, score in results["points"].items():
    combined_scores[name] += score

for name, score in results["normalized_time_diff"].items():
    combined_scores[name] += score

sorted_scores = dict(sorted(combined_scores.items(), key=lambda item: item[1], reverse=True))

In [34]:
def top10(data):
    data = iter(data)
    for i in range(100):
        row = next(data)
        print(f"{i+1} - {row}")
top10(sorted_scores)

1 - FRANTZ Nicolas
2 - LEDUCQ André
3 - THYS Philippe
4 - GARRIGOU Gustave
5 - FABER François
6 - ALAVOINE Jean
7 - MERCKX Eddy
8 - MAGNE Antonin
9 - CHRISTOPHE Eugène
10 - ZABEL Erik
11 - SAGAN Peter
12 - ZOETEMELK Joop
13 - PETIT-BRETON Lucien
14 - BOTTECCHIA Ottavio
15 - DARRIGADE André
16 - KELLY Sean
17 - PÉLISSIER Henri
18 - POGAČAR Tadej
19 - HINAULT Bernard
20 - POULIDOR Raymond
21 - GEORGET Émile
22 - INDURÁIN Miguel
23 - DEFRAEYE Odiel
24 - PÉLISSIER Charles
25 - CAVENDISH Mark
26 - MOTTIAT Louis
27 - BELLENGER Romain
28 - LAMBOT Firmin
29 - TROUSSELIER Louis
30 - VAN IMPE Lucien
31 - JANSSEN Jan
32 - BUYSSE Marcel
33 - LAPIZE Octave
34 - TIBERGHIEN Hector
35 - ROSSIUS Jean
36 - SPEICHER Georges
37 - OCKERS Stan
38 - VERVAECKE Félicien
39 - BUYSSE Lucien
40 - MAES Sylvère
41 - VAN AERT Wout
42 - DEWAELE Maurice
43 - REBRY Gaston
44 - MCEWEN Robbie
45 - DELGADO Pedro
46 - AERTS Jean
47 - ARCHAMBAUD Maurice
48 - ANQUETIL Jacques
49 - AGOSTINHO Joaquim
50 - SELLIER Félix
51 - LE

In [6]:
top_names = {mode: list(results[mode].keys())[:2000] for mode in modes}

best_result = {mode: list(results[mode].values())[0] for mode in modes}
worst_result = {mode: list(results[mode].values())[2000] for mode in modes}

keep_nodes = {mode: { node for node, data in graphs[mode].nodes(data=True) if data.get("label") in top_names[mode] } for mode in modes}

G_top = {mode: graphs[mode].subgraph(keep_nodes[mode]).copy() for mode in modes}

In [9]:
import igraph as ig
import leidenalg
import networkx as nx
from collections import defaultdict

def sizeMap(node,max,min,high,low):
    perc = (node - min) / (max - min)
    return (perc ** 2) * (high - low) + low

def multidigraph_to_digraph(G_multi):
    G_di = nx.DiGraph()
    
    for n, attr in G_multi.nodes(data=True):
        G_di.add_node(n, **attr)

    edge_attr_agg = defaultdict(lambda: {"weight": 0})
    max_weight = 0

    for u, v, data in G_multi.edges(data=True):
        edge_key = (u, v)
        weight = data.get("weight", 1)
        edge_attr_agg[edge_key]["weight"] += weight

        if edge_attr_agg[edge_key]["weight"] > max_weight:
            max_weight = edge_attr_agg[edge_key]["weight"]

        for k, v_attr in data.items():
            if k != "weight":
                edge_attr_agg[edge_key][k] = v_attr

    for (u, v), attr in edge_attr_agg.items():
        G_di.add_edge(u, v, **attr)

    return G_di, max_weight

digraph = dict()
maxWeight = dict()

for mode in modes:
    nodes = list(G_top[mode].nodes())
    idx_map = {n: i for i, n in enumerate(nodes)}
    edges_idx = [(idx_map[u], idx_map[v]) for u, v in G_top[mode].edges()]

    ig_g = ig.Graph(n=len(nodes), edges=edges_idx, directed=G_top[mode].is_directed())
    ig_g.vs['name'] = nodes

    partition_type = leidenalg.RBConfigurationVertexPartition
    
    partition = leidenalg.find_partition(
        ig_g,
        partition_type,
        resolution_parameter=1.0
    )
    membership = partition.membership

    for i, (node, data) in enumerate(G_top[mode].nodes(data=True)):
        G_top[mode].nodes[node]["group"] = membership[i]    
        G_top[mode].nodes[node]["size"] = sizeMap(results[mode][data["label"]], best_result[mode], worst_result[mode], 18, 3)
    digraph[mode], maxWeight[mode] = multidigraph_to_digraph(G_top[mode])

In [12]:
from pyvis.network import Network

for mode in modes:
    pos = nx.fruchterman_reingold_layout(digraph[mode])
    nt = Network('900px', '1500px', notebook=True)
    nt.toggle_physics(False)

    print("Layout calculated")

    for node in digraph[mode].nodes(data=True):
        n_id = node[0]
        n_data = node[1]
        x, y = pos[n_id]
        nt.add_node(
            n_id,
            label = str(n_data["label"]) if n_data.get("label") in list(results["points"])[0:30] else "",
            x = x * 820,  # scale to make layout visible
            y = y * 820,
            title = n_data["label"],
            group = n_data["group"],
            size = n_data["size"],
            fixed = True  # lock the position
        )
    
    print("Node positions calculated")

    edge_list = []
    for u, v, data in digraph[mode].edges(data=True):
        weight = data.get("weight", 1)
        alpha = min(1.0, weight / maxWeight[mode])

        edge_list.append({
            "from": u,
            "to": v,
            "width": 0.5,
            "color": f"rgba(70, 70, 70, {alpha})"
        })

    nt.edges = edge_list

    print("Edges calculated")

    nt.show(f'nx-{mode}.html')

Layout calculated
Node positions calculated
Edges calculated
nx-no_weights.html
Layout calculated
Node positions calculated
Edges calculated
nx-time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-normalized_time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-scaled_time_diff.html
Layout calculated
Node positions calculated
Edges calculated
nx-points.html
Layout calculated
Node positions calculated
Edges calculated
nx-pure_points.html
